



## Introducción a la Programación Usando Modelos **Generativos**
<div style="border-style:groove;border-width:thin;padding:10px">

**Bibliografía:** Hands-On Generative AI with Transformers and Diffusion Models

#### Transformers y Modelos de Lenguaje, cómo funcionan por dentro
</div>

#### Fine tuning
<div style="border-style:groove;border-width:thin;padding:10px">

El fine-tuning (ajuste fino) es el proceso de tomar un modelo de lenguaje preentrenado y reentrenarlo con datos específicos para que se especialice en una tarea o dominio concreto.

</div>

#### Fine tuning ¿Cómo funciona?
<div style="border-style:groove;border-width:thin;padding:10px">

1. Modelo base → ya entrenado con enormes cantidades de texto general (GPT, Claude, Llama, etc.)
2. Dataset propio → pares de ejemplos input/output relevantes para tu caso de uso
3. Reentrenamiento → el modelo ajusta sus pesos internos con tus datos, generalmente con una tasa de aprendizaje baja para no "olvidar" lo que ya sabía
4. Modelo especializado → resultado final adaptado a tu dominio

</div>

#### Fine tuning en generación de texto
<div style="border-style:groove;border-width:thin;padding:10px">

A diferencia de la clasificación (donde las etiquetas son categorías fijas como "Deportes" o "Tecnología"), entrenar un modelo generativo consiste en predecir el siguiente token, donde las salidas son texto libre.
Entrenar desde cero es posible —por ejemplo, con un gran dataset de código— pero requiere semanas o meses de cómputo, lo que lo hace inviable para la mayoría de proyectos.
La alternativa más práctica es hacer fine-tuning de un modelo existente para que genere texto en un estilo específico. Así se aprovecha el conocimiento previo del modelo sobre el lenguaje, reduciendo drásticamente la cantidad de datos y potencia de cómputo necesaria.

Ejemplo concreto: bastarían unos pocos cientos de tus propios tweets para entrenar un modelo que genere nuevos tweets imitando tu estilo de escritura.

</div>

#### Escoger el modelo adecuado: Factores clave
<div style="border-style:groove;border-width:thin;padding:10px">

<b>Tamaño del modelo</b>

No siempre más grande es mejor. Depende del hardware disponible, el tiempo de inferencia aceptable y los requisitos de despliegue.

<b>Datos de entrenamiento</b>

El modelo base debería haber sido entrenado con datos similares a tu caso de uso. 

<b>Longitud de contexto</b>

Cada modelo tiene un límite de tokens que puede "recordar". Para generar textos largos, necesitas un modelo con contexto amplio.

<b>Licencia</b>

Fundamental antes de usar cualquier modelo. Hay licencias comerciales, no comerciales, open source y open access. Algunas incluso restringen cómo puedes usar las salidas del modelo (por ejemplo, prohiben usarlas para entrenar otro modelo).

</div>

#### Artificial Analysis Leaderboard

https://artificialanalysis.ai/leaderboards/models


#### Hugging Face Open LLM Leaderboard 

https://huggingface.co/open-llm-leaderboard

#### Escoger el modelo adecuado: Benchmarks de Hugging Face
<div style="border-style:groove;border-width:thin;padding:10px">

Benchmarks del LLM Leaderboard de Hugging Face

- MMLU-Pro: Conocimiento general
- GPQA: Conocimiento científico
- MuSR: Razonamiento en múltiples pasos
- MATH: Resolución de problemas matemáticos difíciles
- BBH: Mezcla de razonamiento, lenguaje y conocimiento
- IFEval: Seguimiento de instrucciones (el único orientado a conversación)

</div>

#### <b>Entrenar un modelo generativo</b>
<div style="border-style:groove;border-width:thin;padding:10px">

Como tenemos limitaciones de hardware y queremos hacer una prueba rápida, vamos a usar un modelo pequeño, SmolLM, de 135 millones de parámetros.

</div>

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch
if torch.cuda.is_available():
    device = "cuda" #la gráfica NVIDIA
elif torch.backends.mps.is_available():
    device = "mps" #Chip de apple (M1,M2,M3)
else:
    device = "cpu" #Al procesador normal

model_id = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(device)


/home/ciabd12/Documentos/Ejercicios-Python/_Entorno/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ciabd12/Documentos/Ejercicios-Python/_Entorno/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


cpu


Cuando procesas varias secuencias a la vez (un batch), estas pueden tener distinta longitud:

Secuencia 1: ["Hola", "mundo"]            → 2 tokens

Secuencia 2: ["El", "gato", "es", "gris"] → 4 tokens

El modelo necesita que todas tengan la misma longitud, así que se rellenan con un token especial llamado padding

Secuencia 1: ["Hola", "mundo",PAD, PAD]

Secuencia 2: ["El",   "gato",  "es",  "gris"]

SmolLM no tiene un pad_token definido. Si intentas hacer padding sin él, obtienes un error.

La solución es reutilizar el eos_token (token de fin de secuencia)

In [4]:
tokenizer.pad_token = (
    tokenizer.eos_token
) # Esta linea es porque SmolLM no tiene un padding específico.
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4629.55it/s]


El siguiente paso sería tokenizar el dataset que vamos a usar para entrenar.
En este ejemplo vamos a tratar de entrenar el modelo para generar noticias empresariales. Para ello, cogemos un dataset de noticias (de hugging face) y dejamos las noticias etiquetadas como "business".


In [5]:
# %pip install datasets
from datasets import load_dataset
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Como vemos tiene dos conjuntos, uno de train y otro de test. La etiqueta corresponde al tipo de noticia. Vamos a ver un dato de train:

In [6]:

raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

Como solo queremos noticias de empresa, nos vamos a quedar con las noticias empresariales:

In [7]:
def filter_by_label(example):
    return example["label"] == 2

filtered_datasets = raw_datasets.filter(filter_by_label)
filtered_datasets = filtered_datasets.remove_columns("label")

Ya filtrado el dataset, nos tocaría tokenizarlo. 

La función tokenize_function recibe un batch (lote) de ejemplos del dataset.
Llama al tokenizador mandando la columna "text" de cada batch.
truncation=True está para recortar los textos más largos de lo que admite el modelo. Esta función devuelve input_ids y attention_mask para cada ejemplo.

El método .map() sirve para aplicar una función a todo el dataset.

batched=True procesa varios ejemplos a la vez en lugar de uno por uno, lo que es mucho más rápido.

remove_columns=["text"] elimina la columna de texto original, ya que una vez tokenizado no la necesitamos. Solo nos quedamos con input_ids y attention_mask.


Recordamos que los input_ids son los tokens convertidos a números. Cada palabra (o fragmento de palabra) del vocabulario del modelo tiene un ID único asignado.

La attention_mask le dice con 1 o 0 a que token debe atender y a cual no. Evita, por ejemplo, que haga caso de los tokens de relleno (padding)

In [8]:
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_datasets = filtered_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"], # We only need the input_ids and attention_mask
)
tokenized_datasets

Map: 100%|██████████| 1900/1900 [00:00<00:00, 36749.22 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1900
    })
})

Ahora vamos a cargar el data collector que es el componente que toma ejemplos individuales del dataset y los agrupa en batches listos para entrenar, gestionando el padding dinámicamente. 



In [9]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,
mlm=False)

tokenizer=tokenizer necesita el tokenizador para saber qué token usar como padding.

mlm es un parámetro que indica el tipo de función que va a hacer nuestro sistema. Existen dos tipos de entrenamiento, MLM (Masked Language Modeling) y CLM (Causal Language Modeling). El MLM es para predecir tokens enmascarados, por ejemplo "El [MASK] es gris"  → predice "gato".

El CLM es el que estamos usando nosotros ahora, el predictor de la siguiente palabra. Por eso ponemos mlm a False.

Veamos como funciona este data_collator:

In [10]:
samples = []
for i in range(3):
    samples.append(tokenized_datasets["train"][i])

for sample in samples:
    print(f"Número de input_ids en la muestra: {len(sample['input_ids'])}")

Número de input_ids en la muestra: 39
Número de input_ids en la muestra: 56
Número de input_ids en la muestra: 51


Vemos que las muestras tienen un número variable de tokens. Eso lo soluciona el data_collator

In [11]:
out = data_collator(samples)
for key in out:
    print(f"{key} shape: {out[key].shape}")

input_ids shape: torch.Size([3, 56])
attention_mask shape: torch.Size([3, 56])
labels shape: torch.Size([3, 56])


Ahora definimos los argumentos del entrenamiento: 

Weight Decay (Regularización de pesos)

Técnica para evitar el overfitting (sobreajuste). Añade una penalización a la función de pérdida para evitar que el modelo asigne pesos demasiado grandes. Ajustarlo correctamente mejora la capacidad del modelo de generalizar a datos nuevos.

Learning Rate (Tasa de aprendizaje)

Parecido a lo que hemos visto hasta ahora.

Learning Rate Scheduler (Planificador de la tasa de aprendizaje)

Define cómo evoluciona la tasa de aprendizaje a lo largo del entrenamiento, en lugar de mantenerla fija. Algunas estrategias comunes son tasa constante, cosine annealing (reducción suave en forma de coseno), entre otras. 


In [12]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    "business-news-generator",
    push_to_hub=False, #Esto es si queremos subir el modelo a hugging face
    per_device_train_batch_size=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=200,
)

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    # processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["test"],
)
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

Si quisieramos compartir el modelo con la comunidad Hugging Face, podríamos hacer trainer.push_to_hub()


Vamos a probar a generar noticias:

In [ ]:
from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    device=device,
)
print(
    pipe("Q1", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
    "generated_text"
    ]
)
print(
    pipe("Wall", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
    "generated_text"
    ]
)
print(
    pipe("Google", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
    "generated_text"
    ]
)

Q1: China #39;s Airline Unions Reject Offer China #39;s Airline Group, the world #39;
Wall Street Seen Flat After Jobless Data  NEW YORK (Reuters) - Wall Street was seen looking flat on  Friday after a report showed
Google IPO Imminent Google #39;s stock price is set for \$85.90, the lowest price yet, and
